# Modélisation - Classes Entités

In [37]:
/**
 * Entité représentant un employé.
 * Contient les informations d'identité et de rattachement hiérarchique.
 */
public class Collaborateur {
    private int idEmploye;
    private String nom;
    private String prenom;
    private int idDepartement;
    private Integer idManager;
    private String poste;

    public Collaborateur(int id, String n, String p, int idDep, Integer idMan, String post) {
        this.setIdEmploye(id);
        this.setNom(n);
        this.setPrenom(p);
        this.idDepartement = idDep;
        this.idManager = idMan;
        this.poste = post;
    }

    public void setIdEmploye(int id) {
        if (id <= 0) throw new IllegalArgumentException("L'ID employé doit être positif.");
        this.idEmploye = id;
    }

    public void setNom(String n) {
        if (n == null || n.length() < 2) throw new IllegalArgumentException("Le nom est trop court.");
        this.nom = n;
    }

    public void setPrenom(String p) {
        if (p == null || p.length() < 2) throw new IllegalArgumentException("Le prénom est trop court.");
        this.prenom = p;
    }

    public int getIdEmploye() { return idEmploye; }
    public String getNom() { return nom; }
    public String getPrenom() { return prenom; }
    public int getIdDepartement() { return idDepartement; }
    public Integer getIdManager() { return idManager; }
    public String getPoste() { return poste; }
}

In [38]:
/**
 * Entité représentant un département de l'organisation.
 * Gère le nom et l'identifiant unique du service.
 */
public class Departement {
    private int idDepartement;
    private String nomDepartement;

    /**
     * Constructeur utilisant les setters pour valider les données.
     */
    public Departement(int id, String nom) {
        this.setIdDepartement(id);
        this.setNomDepartement(nom);
    }

    public void setIdDepartement(int id) {
        if (id <= 0) throw new IllegalArgumentException("L'ID département doit être positif.");
        this.idDepartement = id;
    }

    public void setNomDepartement(String nom) {
        if (nom == null || nom.trim().isEmpty()) throw new IllegalArgumentException("Le nom du département ne peut pas être vide.");
        this.nomDepartement = nom;
    }

    public int getIdDepartement() { return idDepartement; }
    public String getNomDepartement() { return nomDepartement; }
}

In [39]:
/**
 * Entité représentant une règle de la politique voyage (Plafonds).
 */
public class Depense {
    private int idDepense;
    private String natureDepense;
    private double plafondMax;
    private String description;

    public Depense(int id, String nature, double plafond, String desc) {
        this.idDepense = id;
        this.natureDepense = nature;
        this.setPlafondMax(plafond);
        this.description = desc;
    }

    public void setPlafondMax(double p) {
        if (p < 0) throw new IllegalArgumentException("Le plafond ne peut pas être négatif.");
        this.plafondMax = p;
    }

    public int getIdDepense() { return idDepense; }
    public String getNatureDepense() { return natureDepense; }
    public double getPlafondMax() { return plafondMax; }
    public String getDescription() { return description; }
}

In [40]:
/**
 * Entité centrale gérant une ligne de frais.
 * Calcule le montant TTC et vérifie la conformité par rapport aux plafonds.
 */
public class NoteDeFrais {
    private int idNote;
    private int idEmploye;
    private String date;
    private int idDepense;
    private double montantHT;
    private double tva;
    private int departementImputation;

    public NoteDeFrais(int idN, int idE, String d, int idD, double ht, double tva, int depImp) {
        this.setIdNote(idN);
        this.idEmploye = idE;
        this.date = d;
        this.idDepense = idD;
        this.setMontantHT(ht);
        this.setTva(tva);
        this.departementImputation = depImp;
    }

    public void setIdNote(int id) {
        if (id <= 0) throw new IllegalArgumentException("ID Note incorrect.");
        this.idNote = id;
    }

    public void setMontantHT(double ht) {
        if (ht < 0) throw new IllegalArgumentException("Montant HT négatif interdit.");
        this.montantHT = ht;
    }

    public void setTva(double t) {
        if (t < 0) throw new IllegalArgumentException("TVA négative interdite.");
        this.tva = t;
    }

    // --- MÉTHODES DE CALCUL ---

    /**
     * @return Somme du HT et de la TVA.
     */
    public double getMontantTTC() {
        return this.montantHT + this.tva;
    }

    /**
     * Compare le TTC au plafond de la politique.
     * @param plafond Le plafond autorisé pour cette nature.
     * @return "FRAUDE" ou "OK".
     */
    public String getStatutAudit(double plafond) {
        return (this.getMontantTTC() > plafond) ? "FRAUDE" : "OK";
    }

    // --- ACCESSEURS ---
    public int getIdNote() { return idNote; }
    public double getMontantHT() { return montantHT; }
    public double getTva() { return tva; }
}

In [41]:
try {
    // 1. Création d'une règle (Repas max 30€)
    Depense regleRepas = new Depense(2, "Repas", 30.0, "Plafond repas");

    // 2. Création d'une note valide
    NoteDeFrais n1 = new NoteDeFrais(101, 7, "12/03/2026", 2, 20.0, 4.0, 3);
    System.out.println("Note 101 - TTC: " + n1.getMontantTTC() + " - Statut: " + n1.getStatutAudit(regleRepas.getPlafondMax()));

    // 3. Test de fraude
    NoteDeFrais n2 = new NoteDeFrais(102, 7, "12/03/2026", 2, 50.0, 10.0, 3);
    System.out.println("Note 102 - TTC: " + n2.getMontantTTC() + " - Statut: " + n2.getStatutAudit(regleRepas.getPlafondMax()));

    // 4. Test de sécurité (doit provoquer une erreur)
    System.out.println("Tentative montant négatif...");
    NoteDeFrais nErr = new NoteDeFrais(103, 7, "12/03/2026", 2, -10.0, 2.0, 3);

} catch (IllegalArgumentException e) {
    System.out.println("ERREUR CAPTURÉE : " + e.getMessage());
}

Note 101 - TTC: 24.0 - Statut: OK
Note 102 - TTC: 60.0 - Statut: FRAUDE
Tentative montant négatif...
ERREUR CAPTURÉE : Montant HT négatif interdit.
